## Experimenting with ypet-dnaN analysis

In [ ]:
import copy
import os
import shutil
import time

import dask
import dask.array as da
import dask.dataframe as dd
import h5py
import holoviews as hv
import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats
import seaborn as sns
import skimage as sk
import sklearn as skl
import xarray as xr
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
from statsmodels.nonparametric import kernel_regression

import paulssonlab.deaton.trenchripper.trenchripper as tr

hv.extension("bokeh")

In [ ]:
# addition of active memory manager
import dask

dask.config.set({"distributed.scheduler.active-memory-manager.start": True})
dask.config.set({"distributed.scheduler.worker-ttl": "5m"})
dask.config.set({"distributed.scheduler.allowed-failures": 100})

dask_wd = "/home/de64/scratch/de64/dask"

### Step 1: Make Projections
- Raw projections only
- Only for one dataset currently, needs to be modified to work with more than one

In [ ]:
# dask_controller = tr.trcluster.dask_controller(
#     walltime="02:00:00",
#     local=False,
#     n_workers=100,
#     n_workers_min=100,
#     memory="8GB",
#     working_directory=dask_wd,
# )
# dask_controller.startdask()

dask_controller = tr.trcluster.dask_controller(
    walltime="02:00:00",
    local=False,
    n_workers=10,
    n_workers_min=10,
    memory="8GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

In [ ]:
dask_controller.daskclient

In [ ]:
### rudamentary fork plot function
headpath = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Growth_Division"
intensity_channel = "mCherry"
seg_key = "fluorsegmentation"
projection_path = headpath + "/Projection_" + intensity_channel

projection_handle = tr.projection_handler(headpath)
projection_handle.export_projection(
    dask_controller, seg_key, intensity_channel, projection_path, overwrite=True
)

### Step 2: Review Output and Test Forkplot

- Notes for next time
- This has too much memory load when lookup_projection_df is called
- This dataframe is huge, even for few trenches. Best to re-export the projections with a length column and re-write downstream code

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="04:00:00",
    local=False,
    n_workers=50,
    n_workers_min=50,
    memory="16GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

In [ ]:
dask_controller.daskclient

### Get Forkplots and Segregation Probabilities

In [ ]:
### the memory effeciency of this function can be optimized during the bootstrap at a later date


def lookup_projection_df(
    projection_dd,
    file_index_list,
    file_trench_index_list,
    time_range=(0, 1000),
    index_precisions=np.array([4, 4, 2, 2, 0], dtype="uint"),
    index_columns=[
        "File Index",
        "File Trench Index",
        "timepoints",
        "Projection Length",
        "Major Axis",
    ],
):
    index_precisions = index_precisions.astype("uint")
    rolling_precisions = np.add.accumulate(index_precisions[::-1])
    precision_factors = 10**rolling_precisions

    index_lookup_list = [
        {
            "File Index": file_index_list[i],
            "File Trench Index": file_trench_index_list[i],
        }
        for i in range(len(file_index_list))
    ]
    projection_df_list = []
    for index_lookup in index_lookup_list:
        init_lower, _ = tr.lookup_parquet_index(
            index_lookup, index_precisions, index_columns
        )
        time_lower_factor = np.array(time_range[0], dtype="uint") * precision_factors[2]
        time_upper_factor = (
            np.array(time_range[1] + 1, dtype="uint") * precision_factors[2]
        )
        lower = init_lower + time_lower_factor
        upper = init_lower + time_upper_factor
        projection_df = (
            projection_dd[
                [
                    "File Index",
                    "File Trench Index",
                    "Projection Index",
                    "Mean",
                    "Projection Length",
                    "timepoints",
                    "Major Axis",
                    "Cell Projection ID",
                ]
            ]
            .loc[lower:upper]
            .compute(scheduler="threads")
        )
        projection_df = projection_df.set_index(
            [
                "File Index",
                "File Trench Index",
                "timepoints",
                "Projection Length",
                "Projection Index",
            ]
        )
        projection_df_list.append(projection_df)

    projection_df = pd.concat(projection_df_list, axis=0)
    projection_df = projection_df.sort_index()
    return projection_df


def get_bootstrap_confidence_interval(x, n_resamples=100):
    bootstrap_output = sp.stats.bootstrap(
        [x], np.nanmean, n_resamples=n_resamples, vectorized=True
    )
    bootstrap_output = bootstrap_output.bootstrap_distribution
    return bootstrap_output


def get_fork_plot_df(
    projection_df,
    projection_name="Mean",
    projection_type="Major",
    min_n_cells=3,
    bootstrap=True,
    n_resamples=100,
):
    if projection_type not in ["Major", "Minor"]:
        raise ValueError("projection_type must be either 'Major' or 'Minor'")
    elif projection_type == "Major":
        working_projection_df = projection_df.loc[1]
    else:
        working_projection_df = projection_df.loc[0]

    final_projection_size = (
        working_projection_df.groupby(
            ["File Index", "File Trench Index", "timepoints", "Cell Projection ID"]
        )
        .size()
        .to_frame()
    )
    final_projection = (
        working_projection_df.groupby(
            ["File Index", "File Trench Index", "timepoints", "Cell Projection ID"]
        )
        .apply(lambda x: np.array(x[projection_name].tolist()))
        .to_frame()
    )
    final_projection_df = pd.concat([final_projection_size, final_projection], axis=1)
    final_projection_df.columns = ["Projection Length", projection_name]
    max_fork_len = final_projection_df["Projection Length"].max()
    final_projection_df["Difference From Max Projection Len"] = (
        max_fork_len - final_projection_df["Projection Length"]
    )
    final_projection_df["Left Pad"] = np.ceil(
        final_projection_df["Difference From Max Projection Len"] / 2
    ).astype(int)
    final_projection_df["Right Pad"] = np.floor(
        final_projection_df["Difference From Max Projection Len"] / 2
    ).astype(int)
    final_projection_df[projection_name] = final_projection_df.apply(
        lambda x: np.pad(
            x[projection_name], (x["Left Pad"], x["Right Pad"]), constant_values=np.NaN
        ),
        axis=1,
    )
    final_projection_df = final_projection_df[[projection_name, "Projection Length"]]

    projection_length_past_thr = (
        final_projection_df.groupby("Projection Length").size() >= min_n_cells
    )
    projection_length_past_thr = projection_length_past_thr[
        projection_length_past_thr
    ].index.tolist()
    final_projection_df_filtered = final_projection_df[
        final_projection_df["Projection Length"].isin(projection_length_past_thr)
    ]

    if len(final_projection_df_filtered) < 1:
        return None

    fork_plot_df = (
        final_projection_df_filtered.groupby("Projection Length")
        .apply(lambda x: np.nanmean(np.stack(x[projection_name], axis=0), axis=0))
        .to_frame()
    )
    fork_plot_df.columns = [projection_name]
    fork_plot_df["N Cells"] = final_projection_df_filtered.groupby(
        "Projection Length"
    ).size()
    fork_plot_df["Projection Index"] = fork_plot_df[projection_name].apply(
        lambda x: list(range(len(x)))
    )
    if bootstrap:
        bootstrap_series = final_projection_df_filtered.groupby(
            "Projection Length"
        ).apply(
            lambda x: get_bootstrap_confidence_interval(
                np.stack(x[projection_name], axis=0), n_resamples=n_resamples
            )
        )
        fork_plot_df["Bootstrap"] = bootstrap_series.apply(
            lambda x: [x[i] for i in range(x.shape[0])]
        )
        fork_plot_df = fork_plot_df.explode(
            [projection_name, "Projection Index", "Bootstrap"]
        )
        fork_plot_df = fork_plot_df.reset_index().set_index(
            ["Projection Length", "Projection Index"]
        )
        fork_plot_df["Bootstrap Index"] = fork_plot_df["Bootstrap"].apply(
            lambda x: list(range(len(x)))
        )
        fork_plot_df = fork_plot_df.explode(["Bootstrap Index", "Bootstrap"])
        fork_plot_df = fork_plot_df.reset_index().set_index(
            ["Projection Length", "Projection Index", "Bootstrap Index"]
        )
        return fork_plot_df
    else:
        fork_plot_df = fork_plot_df.explode([projection_name, "Projection Index"])
        fork_plot_df = fork_plot_df.reset_index().set_index(
            ["Projection Length", "Projection Index"]
        )
        return fork_plot_df


def get_fork_plot_arr(fork_plot_df):
    fork_plot_xarr = fork_plot_df.to_xarray()
    fork_plot_dataarr = fork_plot_xarr.to_array()
    fork_plot_arr = fork_plot_dataarr.data.astype(float)[0]

    ## truncate hanging NaNs
    nan_range = np.all(np.isnan(fork_plot_arr), axis=0)
    fork_plot_arr = fork_plot_arr[:, ~nan_range]

    return fork_plot_arr


def scale_and_export_forkplot(
    projection_path,
    scaling_path,
    variant_idx,
    variant_col_name,
    file_index_list,
    file_trench_index_list,
    rescaling_column,
    time_range,
    min_n_cells,
    n_resamples=1000,
    time_read_step=12,
    bootstrap=True,
):
    min_time, max_time = time_range

    scaling_df = pd.read_pickle(scaling_path)
    scaling_df_unique_indices = list(scaling_df.index.unique())
    scaling_df_index_subset = sorted(
        list(
            set(zip(file_index_list, file_trench_index_list))
            & set(scaling_df_unique_indices)
        )
    )
    scaling_df = scaling_df.loc[scaling_df_index_subset]
    scaling_df = scaling_df[["timepoints", rescaling_column]]
    scaling_df = scaling_df.reset_index().set_index(
        ["File Index", "File Trench Index", "timepoints"]
    )

    projection_dd = dd.read_parquet(
        projection_path, calculate_divisions=True, scheduler="threads"
    )
    projection_df = []
    for init_time in range(min_time, max_time + 1, time_read_step):
        fin_time = init_time + time_read_step
        print((init_time, fin_time))
        working_projection_df = lookup_projection_df(
            projection_dd,
            file_index_list,
            file_trench_index_list,
            time_range=(init_time, fin_time),
        )
        working_projection_df = working_projection_df.loc[
            (slice(None), slice(None), slice(min_time, max_time + 1)), :
        ]
        projection_df.append(working_projection_df)
    projection_df = pd.concat(projection_df, axis=0)
    ## using slices saves memory

    projection_df = projection_df.join(
        scaling_df[rescaling_column],
        how="left",
        on=["File Index", "File Trench Index", "timepoints"],
    )
    projection_df = projection_df.dropna(axis=0, subset=rescaling_column)
    projection_df["Rescaled Mean"] = (
        projection_df["Mean"] * projection_df[rescaling_column]
    )
    del projection_df["Mean"]
    del projection_df[rescaling_column]

    projection_df = (
        projection_df.reset_index()
        .set_index(
            [
                "Major Axis",
                "File Index",
                "File Trench Index",
                "timepoints",
                "Cell Projection ID",
            ]
        )
        .sort_index()
    )
    fork_plot_df = get_fork_plot_df(
        projection_df,
        projection_name="Rescaled Mean",
        projection_type="Major",
        min_n_cells=min_n_cells,
        bootstrap=bootstrap,
        n_resamples=n_resamples,
    )
    if bootstrap:
        if fork_plot_df is None:
            fork_plot_df = pd.DataFrame(
                data=[],
                columns=[
                    variant_col_name,
                    "Projection Length",
                    "Projection Index",
                    "Bootstrap Index",
                    "Rescaled Mean",
                    "N Cells",
                    "Bootstrap",
                ],
            )
            fork_plot_df = fork_plot_df.astype(
                {
                    variant_col_name: int,
                    "Projection Length": int,
                    "Projection Index": int,
                    "Bootstrap Index": int,
                    "Bootstrap": float,
                    "Rescaled Mean": float,
                    "N Cells": int,
                }
            )
            fork_plot_df = fork_plot_df.set_index(variant_col_name)
            return fork_plot_df

        fork_plot_df[variant_col_name] = variant_idx
        fork_plot_df = (
            fork_plot_df.reset_index()
            .set_index(
                [
                    variant_col_name,
                    "Projection Length",
                    "Projection Index",
                    "Bootstrap Index",
                ]
            )
            .sort_index()
        )
        fork_plot_df = fork_plot_df.reset_index()
        fork_plot_df = fork_plot_df.set_index(variant_col_name)
        fork_plot_df["Bootstrap"] = fork_plot_df["Bootstrap"].astype(float)
        fork_plot_df["Rescaled Mean"] = fork_plot_df["Rescaled Mean"].astype(float)

    else:
        if fork_plot_df is None:
            fork_plot_df = pd.DataFrame(
                data=[],
                columns=[
                    variant_col_name,
                    "Projection Length",
                    "Projection Index",
                    "Rescaled Mean",
                    "N Cells",
                ],
            )
            fork_plot_df = fork_plot_df.astype(
                {
                    variant_col_name: int,
                    "Projection Length": int,
                    "Projection Index": int,
                    "Rescaled Mean": float,
                    "N Cells": int,
                }
            )
            fork_plot_df = fork_plot_df.set_index(variant_col_name)
            return fork_plot_df

        fork_plot_df[variant_col_name] = variant_idx
        fork_plot_df = (
            fork_plot_df.reset_index()
            .set_index([variant_col_name, "Projection Length", "Projection Index"])
            .sort_index()
        )
        fork_plot_df = fork_plot_df.reset_index()
        fork_plot_df = fork_plot_df.set_index(variant_col_name)
        fork_plot_df["Rescaled Mean"] = fork_plot_df["Rescaled Mean"].astype(float)

    return fork_plot_df


def lookup_midpoint_mean_df(
    midpoint_mean_dd, file_index_list, file_trench_index_list, time_range=(0, 1000)
):
    index_precisions = np.array([4, 4, 2, 2], dtype="uint")
    index_columns = [
        "File Index",
        "File Trench Index",
        "timepoints",
        "Cell Projection ID",
    ]

    index_precisions = index_precisions.astype("uint")
    rolling_precisions = np.add.accumulate(index_precisions[::-1])
    precision_factors = 10**rolling_precisions

    index_lookup_list = [
        {
            "File Index": file_index_list[i],
            "File Trench Index": file_trench_index_list[i],
        }
        for i in range(len(file_index_list))
    ]
    midpoint_mean_df_list = []
    for index_lookup in index_lookup_list:
        init_lower, _ = tr.lookup_parquet_index(
            index_lookup, index_precisions, index_columns
        )
        time_lower_factor = np.array(time_range[0], dtype="uint") * precision_factors[1]
        time_upper_factor = (
            np.array(time_range[1] + 1, dtype="uint") * precision_factors[1]
        )
        lower = init_lower + time_lower_factor
        upper = init_lower + time_upper_factor
        midpoint_mean_df = (
            midpoint_mean_dd[
                [
                    "File Index",
                    "File Trench Index",
                    "timepoints",
                    "Cell Projection ID",
                    "Midpoint Mean",
                    "Projection Length",
                    "Length",
                ]
            ]
            .loc[lower:upper]
            .compute(scheduler="threads")
        )
        midpoint_mean_df = midpoint_mean_df.set_index(
            [
                "File Index",
                "File Trench Index",
                "timepoints",
                "Projection Length",
                "Cell Projection ID",
            ]
        )
        midpoint_mean_df_list.append(midpoint_mean_df)

    midpoint_mean_df = pd.concat(midpoint_mean_df_list, axis=0)
    midpoint_mean_df = midpoint_mean_df.sort_index()
    return midpoint_mean_df


def get_midpoint_value(x, proj_value="Rescaled Mean", center_width=2):
    try:
        signal = x.set_index("Projection Index").sort_index()[proj_value].values
        peaks = sp.signal.find_peaks(signal)[0]
        midpoint = int(np.mean(peaks))
        midpoint_mean = np.mean(
            signal[midpoint - center_width : midpoint + center_width]
        )
    except:
        midpoint_mean = np.NaN
    return midpoint_mean


def scale_and_export_midcell_intensity(
    file_idx, projection_path, scaling_path, rescaling_column
):
    index_precisions = np.array([4, 4, 2, 2], dtype="uint")
    index_columns = [
        "File Index",
        "File Trench Index",
        "timepoints",
        "Cell Projection ID",
    ]

    scaling_df = pd.read_pickle(scaling_path)
    scaling_df = scaling_df.loc[file_idx:file_idx]
    scaling_df = scaling_df[["timepoints", rescaling_column]]
    scaling_df = scaling_df.reset_index().set_index(
        ["File Index", "File Trench Index", "timepoints"]
    )

    projection_dd = dd.read_parquet(
        projection_path, calculate_divisions=True, scheduler="threads"
    )
    lower_idx, upper_idx = tr.lookup_parquet_index(
        {"File Index": file_idx}, index_precisions, index_columns
    )
    projection_df = projection_dd.loc[lower_idx:upper_idx].compute(scheduler="threads")
    projection_df = projection_df[projection_df["Major Axis"] == 1]

    projection_df = projection_df.join(
        scaling_df[rescaling_column],
        how="left",
        on=["File Index", "File Trench Index", "timepoints"],
    )
    projection_df = projection_df.dropna(axis=0, subset=rescaling_column)
    projection_df["Rescaled Mean"] = (
        projection_df["Mean"] * projection_df[rescaling_column]
    )
    del projection_df["Mean"]
    del projection_df[rescaling_column]

    midpoint_mean_series = projection_df.groupby(
        ["File Index", "File Trench Index", "timepoints", "Cell Projection ID"]
    ).apply(lambda x: get_midpoint_value(x))
    cell_proj_length_series = projection_df.groupby(
        ["File Index", "File Trench Index", "timepoints", "Cell Projection ID"]
    ).apply(lambda x: x["Projection Length"].iloc[0])
    cell_length_series = projection_df.groupby(
        ["File Index", "File Trench Index", "timepoints", "Cell Projection ID"]
    ).apply(lambda x: x["Length"].iloc[0])

    midpoint_mean_df = pd.concat(
        [midpoint_mean_series, cell_proj_length_series, cell_length_series], axis=1
    )
    midpoint_mean_df = midpoint_mean_df.rename(
        columns={0: "Midpoint Mean", 1: "Projection Length", 2: "Length"}
    ).reset_index()

    midpoint_mean_df["Projection File Index"] = tr.make_parquet_index(
        midpoint_mean_df, index_columns, index_precisions
    )
    midpoint_mean_df = midpoint_mean_df.set_index("Projection File Index").sort_index()

    return midpoint_mean_df


def get_segregation_prob(
    midcell_mean_path,
    variant_idx,
    variant_col_name,
    file_index_list,
    file_trench_index_list,
    time_range,
    pixel_microns,
    segregation_thr,
    prob_segregation_thr,
    min_n_cells,
    max_intensity_sem,
    kernel_bandwidth,
    n_resamples,
    time_read_step=12,
):
    min_time, max_time = time_range

    midcell_mean_dd = dd.read_parquet(
        midcell_mean_path, calculate_divisions=True, scheduler="threads"
    )
    midcell_mean_df = []
    for init_time in range(min_time, max_time + 1, time_read_step):
        fin_time = init_time + time_read_step
        print((init_time, fin_time))
        working_midpoint_mean_df = lookup_midpoint_mean_df(
            midcell_mean_dd,
            file_index_list,
            file_trench_index_list,
            time_range=(init_time, fin_time),
        )
        working_midpoint_mean_df = working_midpoint_mean_df.loc[
            (slice(None), slice(None), slice(min_time, max_time + 1)), :
        ]
        midcell_mean_df.append(working_midpoint_mean_df)
    midcell_mean_df = pd.concat(midcell_mean_df, axis=0)
    segregated_prob_df = get_segregated_prob_df(
        midcell_mean_df,
        pixel_microns,
        segregation_thr=segregation_thr,
        prob_segregation_thr=prob_segregation_thr,
        min_n_cells=min_n_cells,
        max_intensity_sem=max_intensity_sem,
        kernel_bandwidth=kernel_bandwidth,
        n_resamples=n_resamples,
    )
    segregated_prob_df[variant_col_name] = variant_idx
    segregated_prob_df = (
        segregated_prob_df.reset_index()
        .set_index([variant_col_name, "Segregation Probability Index"])
        .sort_index()
    )
    return segregated_prob_df


def get_segregated_prob_df(
    midcell_mean_df,
    pixel_microns,
    segregation_thr=0.5,
    prob_segregation_thr=0.5,
    min_n_cells=50,
    max_intensity_sem=300,
    kernel_bandwidth=0.2,
    kernel_steps=100,
    n_resamples=1000,
):
    midpoint_signal_df = (
        midcell_mean_df.reset_index()[["Projection Length", "Midpoint Mean", "Length"]]
        .set_index("Projection Length")
        .sort_index()
    )
    projection_length_groupby = midpoint_signal_df[["Midpoint Mean"]].groupby(
        "Projection Length"
    )
    sem_series = projection_length_groupby.apply(
        lambda x: sp.stats.bootstrap(
            (x.values[:, 0],), np.mean, n_resamples=n_resamples
        ).standard_error
        if len(x.values.T[0]) > 1
        else np.NaN
    )
    sem_series.name = "SEM: Midpoint Mean"
    midpoint_mean_df = pd.concat(
        [
            midpoint_signal_df[["Midpoint Mean"]].groupby("Projection Length").mean(),
            sem_series,
            midpoint_signal_df[["Midpoint Mean"]]
            .groupby("Projection Length")
            .count()
            .rename(columns={"Midpoint Mean": "N Cells"}),
        ],
        axis=1,
    )
    well_sampled_midpoint_df = midpoint_mean_df[
        (midpoint_mean_df["N Cells"] >= min_n_cells)
        & (midpoint_mean_df["SEM: Midpoint Mean"] < max_intensity_sem)
    ]
    try:
        intervals = np.array(well_sampled_midpoint_df.index.tolist()) * pixel_microns
        smoothed_midpoint_mean = kernel_regression.KernelReg(
            well_sampled_midpoint_df["Midpoint Mean"].values,
            intervals,
            "c",
            reg_type="ll",
            bw=np.array([kernel_bandwidth]),
            ckertype="gaussian",
        ).fit(intervals)[0]

        init_val, fin_val = smoothed_midpoint_mean[0], smoothed_midpoint_mean[-1]
        midpoint_signal_df["Midpoint Mean Normalized"] = midpoint_signal_df[
            "Midpoint Mean"
        ].apply(lambda x: (x - fin_val) / (init_val - fin_val))
        midpoint_signal_df["Scaled Length"] = (
            midpoint_signal_df["Length"] * pixel_microns
        )
        well_sampled_lengths = (
            midpoint_signal_df.loc[well_sampled_midpoint_df.index]
            .groupby("Projection Length")["Scaled Length"]
            .mean()
        )
        well_sampled_length_intervals = np.linspace(
            well_sampled_lengths.tolist()[0],
            well_sampled_lengths.tolist()[-1],
            num=kernel_steps,
        )

        midpoint_signal_df_nona = midpoint_signal_df.dropna(
            subset=["Midpoint Mean Normalized", "Scaled Length"]
        )
        midpoint_signal_df_nona["Segregated"] = (
            midpoint_signal_df_nona["Midpoint Mean Normalized"] < segregation_thr
        ).astype(int)
        prob_seg_mean = kernel_regression.KernelReg(
            midpoint_signal_df_nona["Segregated"].values,
            midpoint_signal_df_nona["Scaled Length"].values,
            "c",
            reg_type="ll",
            bw=np.array([kernel_bandwidth]),
            ckertype="gaussian",
        ).fit(well_sampled_length_intervals)[0]
        l_seg = well_sampled_length_intervals[
            np.where(prob_seg_mean > prob_segregation_thr)[0][0]
        ]

        prob_seg_mean_samples = []
        l_seg_samples = []
        for sample_idx in range(n_resamples):
            midpoint_signal_df_nona_sample = midpoint_signal_df_nona.sample(
                frac=1, replace=True
            )
            prob_seg_mean_sample = kernel_regression.KernelReg(
                midpoint_signal_df_nona_sample["Segregated"].values,
                midpoint_signal_df_nona_sample["Scaled Length"].values,
                "c",
                reg_type="ll",
                bw=np.array([kernel_bandwidth]),
                ckertype="gaussian",
            ).fit(well_sampled_length_intervals)[0]
            l_seg_sample = well_sampled_length_intervals[
                np.where(prob_seg_mean_sample > prob_segregation_thr)[0][0]
            ]

            prob_seg_mean_samples.append(prob_seg_mean_sample)
            l_seg_samples.append(l_seg_sample)

        prob_seg_mean_samples = np.stack(prob_seg_mean_samples)
        prob_seg_mean_sem = np.std(prob_seg_mean_samples, axis=0)
        l_seg_sem = np.std(l_seg_samples, axis=0)

        prob_seg_df = pd.DataFrame(
            {
                "Probability Segregated": prob_seg_mean,
                "SEM: Probability Segregated": prob_seg_mean_sem,
                "Length": well_sampled_length_intervals,
            }
        )
        prob_seg_df["Segregation Length"] = l_seg
        prob_seg_df["SEM: Segregation Length"] = l_seg_sem
        prob_seg_df["Segregation Probability Index"] = list(range(len(prob_seg_df)))
        prob_seg_df = prob_seg_df.set_index("Segregation Probability Index")

    except:
        prob_seg_df = pd.DataFrame(
            {
                "Probability Segregated": [],
                "SEM: Probability Segregated": [],
                "Length": [],
                "Segregation Length": [],
                "SEM: Segregation Length": [],
                "Segregation Probability Index": [],
            },
            dtype=float,
        )
        prob_seg_df["Segregation Probability Index"] = prob_seg_df[
            "Segregation Probability Index"
        ].astype(int)
        prob_seg_df = prob_seg_df.set_index("Segregation Probability Index")
        prob_seg_df = prob_seg_df[
            [
                "Probability Segregated",
                "SEM: Probability Segregated",
                "Length",
                "Segregation Length",
                "SEM: Segregation Length",
            ]
        ]

    return prob_seg_df

### Forkplots and Midpoint Means

In [ ]:
### minimal parameter set
headpath = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Growth_Division"
barcodes_df_path = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_lDE26_Final_Barcodes_df_Merged"
midcell_mean_path = (
    "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Midpoint_Mean"
)

fov_time_norm_path_list = [
    "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Growth_Division/2023-06-16_lDE26_Final_FOV_and_Time_Normalization"
]
intensity_channel = "mCherry"
scaling_path = headpath + "/Scaling_" + intensity_channel + ".pkl"
forkplot_df_path = headpath + "/Forkplot_" + intensity_channel + ".pkl"
projection_path = headpath + "/Projection_" + intensity_channel
rescaling_column = "Nucleoid Mean Intensity Mean Rescaling"
segregation_path = headpath + "/Segregation_Probability.pkl"
metadata_path = headpath + "/metadata.hdf5"

variant_col_name = "oDEPool7_id"

forkplot_time_range = (49, 60)
segregation_length_time_range = (49, 60)
kernel_bandwidth = 0.2

min_n_cells_forkplot = 3
n_cells_wt_plot = 1000  # for segplot
n_cells_l_seg = 50

n_resamples_l_seg = 1000
max_intensity_sem_l_seg = 300
prob_segregation_thr = 0.5
segregation_thr = 0.4

In [ ]:
## this should be outside of the parallelization loop

### Fetching Scaling Values For Projection Channel
fov_time_norm_dd_list = [
    dd.read_parquet(path, calculate_divisions=True) for path in fov_time_norm_path_list
]

### Building scaling dataframe based on scalings computed from standard CRISPRi analysis notebook 2
scaling_df_list = []
for fov_time_norm_dd in fov_time_norm_dd_list:
    fov_time_norm_dd["Experiment-file-fileidx-timepoints"] = tr.make_parquet_index(
        fov_time_norm_dd,
        ["Experiment #", "File Index", "File Trench Index", "timepoints"],
        np.array([1, 6, 3, 4]),
    ).persist()
    fov_time_norm_dd = tr.set_new_aligned_index(
        fov_time_norm_dd, "Experiment-file-fileidx-timepoints"
    )
    scaling_df = (
        fov_time_norm_dd.groupby("Experiment-file-fileidx-timepoints")[
            [
                "Experiment #",
                "File Index",
                "File Trench Index",
                "fov",
                "trenchid",
                "timepoints",
                rescaling_column,
            ]
        ]
        .first()
        .compute()
    )
    scaling_df_list.append(scaling_df)
scaling_df = pd.concat(scaling_df_list)
scaling_df = scaling_df.set_index(["File Index", "File Trench Index"])
scaling_df.to_pickle(scaling_path)
file_indices = list(scaling_df.index.get_level_values(0).unique())

### Getting trench indices for each genetic variant
### Filtering out trenches based on notebook 5
barcode_dd = dd.read_parquet(barcodes_df_path, calculate_divisions=True)
barcode_df = (
    barcode_dd.groupby("oDEPool7_id")
    .apply(
        lambda x: sorted(list(set(tuple(zip(x["File Index"], x["File Trench Index"])))))
    )
    .compute()
    .sort_index()
)
barcode_df = barcode_df.apply(lambda x: eval(x))
barcode_df_indices = barcode_df.index.tolist()

### building fork plot dataframe
fork_plot_dd = []
bootstrap_fork_plot_dd = []
for variant_idx in barcode_df_indices:
    working_barcode_entry = barcode_df.loc[variant_idx]
    file_index_list = [item[0] for item in working_barcode_entry]
    file_trench_index_list = [item[1] for item in working_barcode_entry]
    fork_plot_dd.append(
        dask.delayed(scale_and_export_forkplot)(
            projection_path,
            scaling_path,
            variant_idx,
            variant_col_name,
            file_index_list,
            file_trench_index_list,
            rescaling_column,
            forkplot_time_range,
            min_n_cells_forkplot,
            bootstrap=False,
        )
    )
fork_plot_dd = dd.from_delayed(fork_plot_dd)
fork_plot_df = fork_plot_dd.compute()
fork_plot_df = fork_plot_df.reset_index().set_index(
    [variant_col_name, "Projection Length", "Projection Index"]
)
fork_plot_df.to_pickle(forkplot_df_path)

### Getting trench indices for each genetic variant
### Filtering out trenches based on notebook 5

midpoint_mean_dd = []
for file_idx in file_indices:
    midpoint_mean_dd.append(
        dask.delayed(scale_and_export_midcell_intensity)(
            file_idx, projection_path, scaling_path, rescaling_column
        )
    )
midpoint_mean_dd = dd.from_delayed(midpoint_mean_dd)

tr.to_parquet_checkpoint(
    dask_controller,
    midpoint_mean_dd,
    midcell_mean_path,
    engine="pyarrow",
    overwrite=True,
)

##here 10/28

### Get Segregation Dataframe

In [ ]:
global_metadata = tr.pandas_hdf5_handler(metadata_path)
pixel_microns = global_metadata.read_df("global", read_metadata=True).metadata[
    "pixel_microns"
]

midpoint_mean_dd = dd.read_parquet(midcell_mean_path, calculate_divisions=True)
early_midpoint_mean_dd = midpoint_mean_dd[midpoint_mean_dd["timepoints"] < 6]
early_midpoint_mean_df = early_midpoint_mean_dd.sample(frac=0.05).compute()

early_midpoint_mean_df = pd.concat(
    [
        early_midpoint_mean_df.groupby("Projection Length")["Midpoint Mean"].mean(),
        early_midpoint_mean_df.groupby("Projection Length")["Midpoint Mean"]
        .count()
        .to_frame()
        .rename(columns={"Midpoint Mean": "N Cells"}),
        early_midpoint_mean_df.groupby("Projection Length")["Length"].mean().to_frame(),
    ],
    axis=1,
)
early_midpoint_mean_df = early_midpoint_mean_df[
    early_midpoint_mean_df["N Cells"] > n_cells_wt_plot
]
early_midpoint_mean_df["Length Scaled"] = (
    early_midpoint_mean_df.reset_index()["Length"] * pixel_microns
).values
init_early_val, fin_early_val = (
    early_midpoint_mean_df["Midpoint Mean"].iloc[0],
    early_midpoint_mean_df["Midpoint Mean"].iloc[-1],
)
early_midpoint_mean_df["Midpoint Mean Normalized"] = (
    early_midpoint_mean_df["Midpoint Mean"] - fin_early_val
) / (init_early_val - fin_early_val)

sns.scatterplot(
    data=early_midpoint_mean_df.reset_index(),
    x="Length Scaled",
    y="Midpoint Mean Normalized",
)
plt.hlines(
    segregation_thr,
    early_midpoint_mean_df["Length Scaled"].min(),
    early_midpoint_mean_df["Length Scaled"].max(),
    colors="black",
    linestyles="--",
    linewidth=2,
)
plt.show()

In [ ]:
barcode_dd = dd.read_parquet(barcodes_df_path, calculate_divisions=True)
barcode_df = (
    barcode_dd.groupby("oDEPool7_id")
    .apply(
        lambda x: sorted(list(set(tuple(zip(x["File Index"], x["File Trench Index"])))))
    )
    .compute()
    .sort_index()
)
barcode_df = barcode_df.apply(lambda x: eval(x))
barcode_df_indices = barcode_df.index.tolist()

### building fork plot dataframe
segregation_dd = []
for variant_idx in barcode_df_indices:
    working_barcode_entry = barcode_df.loc[variant_idx]
    file_index_list = [item[0] for item in working_barcode_entry]
    file_trench_index_list = [item[1] for item in working_barcode_entry]
    segregation_dd.append(
        dask.delayed(get_segregation_prob)(
            midcell_mean_path,
            variant_idx,
            variant_col_name,
            file_index_list,
            file_trench_index_list,
            segregation_length_time_range,
            pixel_microns,
            segregation_thr,
            prob_segregation_thr,
            n_cells_l_seg,
            max_intensity_sem_l_seg,
            kernel_bandwidth,
            n_resamples_l_seg,
        )
    )
segregation_dd = dd.from_delayed(segregation_dd)
segregation_df = segregation_dd.compute()
segregation_df.to_pickle(segregation_path)

In [ ]:
# barcode_dd = dd.read_parquet(barcodes_df_path,calculate_divisions=True)
# barcode_df = barcode_dd.groupby('oDEPool7_id').apply(lambda x: sorted(list(set(tuple(zip(x['File Index'],x['File Trench Index'])))))).compute().sort_index()
# barcode_df = barcode_df.apply(lambda x: eval(x))
# barcode_df_indices = barcode_df.index.tolist()

### Visualizations and Plots
3. Random cells with nucleoids at the critical segregation values

In [ ]:
headpath = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Growth_Division"

## forkplots with birth, division, segregation length
sgrna_df_path = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/2023-04-25_sgRNA_Timeseries_df.pkl"
segregation_path = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Growth_Division/Segregation_Probability.pkl"
forkplot_df_path = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Growth_Division/Forkplot_mCherry.pkl"
midcell_mean_path = (
    "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Midpoint_Mean"
)
metadata_path = headpath + "/metadata.hdf5"
barcodes_df_path = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_lDE26_Final_Barcodes_df_Merged"
lineage_dd_path = "/home/de64/scratch/de64/sync_folder/2023-03-11_lDE26_Run_1/Growth_Division/2023-06-16_lDE26_Merged_Lineage"

sgrna_df = pd.read_pickle(sgrna_df_path)
segregation_full_df = pd.read_pickle(segregation_path)
segregation_df = segregation_full_df.groupby("oDEPool7_id").first()[
    ["Segregation Length", "SEM: Segregation Length"]
]
sgrna_df = sgrna_df.join(segregation_df)
forkplot_df = pd.read_pickle(forkplot_df_path)
barcode_dd = dd.read_parquet(barcodes_df_path, calculate_divisions=True)
lineage_dd = dd.read_parquet(lineage_dd_path, calculate_divisions=True)

global_metadata = tr.pandas_hdf5_handler(metadata_path)
pixel_microns = global_metadata.read_df("global", read_metadata=True).metadata[
    "pixel_microns"
]

n_cells_wt_plot = 1000
midcell_mean_df_sample = dd.read_parquet(midcell_mean_path).sample(frac=0.01).compute()
projection_length_map = midcell_mean_df_sample.groupby("Projection Length")[
    "Length"
].mean()
projection_length_map_mask = (
    midcell_mean_df_sample.groupby("Projection Length")["Length"].count()
    > n_cells_wt_plot
)
projection_length_reg = skl.linear_model.LinearRegression().fit(
    projection_length_map[projection_length_map_mask].values[:, np.newaxis],
    projection_length_map[projection_length_map_mask].index.values,
)

#### Forkplots

In [ ]:
kernel_label_list = []
zscore_kernel_label_list = [
    item + ": Transformed: z score" for item in kernel_label_list
]
other_kernel_labels = ["Kernel Trace: Division: Length", "Kernel Trace: Birth: Length"]

time_slices = [17]
time_slice_labels = ["7 Hours"]

for time_slice_idx, time_slice in enumerate(time_slices):
    for kernel_label in kernel_label_list:
        zscore_kernel_label = kernel_label + ": Transformed: z score"
        ## to avoid df fragmenation warning
        sgrna_df = sgrna_df.copy()
        ##
        sgrna_df.loc[
            :, time_slice_labels[time_slice_idx] + ": " + kernel_label
        ] = sgrna_df[kernel_label].apply(lambda x: x[time_slice])

        sgrna_df.loc[
            :, time_slice_labels[time_slice_idx] + ": " + zscore_kernel_label
        ] = sgrna_df[zscore_kernel_label].apply(lambda x: x[time_slice])
    for kernel_label in other_kernel_labels:
        sgrna_df.loc[
            :, time_slice_labels[time_slice_idx] + ": " + kernel_label
        ] = sgrna_df[kernel_label].apply(lambda x: x[time_slice])
sgrna_df_subset = sgrna_df[
    [
        "Gene",
        "Segregation Length",
        "7 Hours: Kernel Trace: Division: Length",
        "7 Hours: Kernel Trace: Birth: Length",
    ]
]
sgrna_df_subset = sgrna_df_subset.rename(
    columns={
        "7 Hours: Kernel Trace: Division: Length": "Division Length",
        "7 Hours: Kernel Trace: Birth: Length": "Birth Length",
    }
)

In [ ]:
def convert_to_pixels(length_series, pixel_microns, projection_length_reg):
    length_series = length_series / pixel_microns
    length_series_nonan = length_series.copy()
    length_series_nonan[np.isnan(length_series)] = 0.0
    length_series_nonan_pixels = projection_length_reg.predict(
        (length_series_nonan).values[:, np.newaxis]
    )
    length_series_nonan_pixels[np.isnan(length_series)] = np.NaN
    return length_series_nonan_pixels


def make_forkplot_img(
    ax, var_forkplot_df, var_sgrna_df, vmin_perc=2, vmax_perc=98, img_pad=3, linewidth=2
):
    proj_len_arr = var_forkplot_df.groupby("Projection Length").first().index.values
    n_proj_lens = len(proj_len_arr)

    lower_y_bound = int(var_sgrna_df["Birth Length Pixels"]) - proj_len_arr[0] - img_pad
    upper_y_bound = var_sgrna_df["Division Length Pixels"] - proj_len_arr[0] + img_pad

    lower_x_bound = (
        proj_len_arr[-1] - int(var_sgrna_df["Division Length Pixels"])
    ) // 2 - img_pad
    upper_x_bound = (
        proj_len_arr[-1]
        - (proj_len_arr[-1] - int(var_sgrna_df["Division Length Pixels"])) // 2
    ) + img_pad

    forkplot_arr = get_fork_plot_arr(var_forkplot_df)
    vmax = np.nanpercentile(
        forkplot_arr[int(lower_y_bound) : int(upper_y_bound)], vmax_perc
    )
    vmin = np.nanpercentile(
        forkplot_arr[int(lower_y_bound) : int(upper_y_bound)], vmin_perc
    )

    ax.set_ylim(upper_y_bound, lower_y_bound)
    ax.set_xlim(lower_x_bound, upper_x_bound)
    ax.imshow(forkplot_arr, vmin=vmin, vmax=vmax)
    ax.hlines(
        var_sgrna_df["Division Length Pixels"] - proj_len_arr[0],
        lower_x_bound,
        upper_x_bound,
        linestyle="--",
        color="black",
        linewidth=linewidth,
    )
    ax.hlines(
        var_sgrna_df["Birth Length Pixels"] - proj_len_arr[0],
        lower_x_bound,
        upper_x_bound,
        linestyle="--",
        color="black",
        linewidth=linewidth,
    )
    ax.hlines(
        var_sgrna_df["Segregation Length Pixels"] - proj_len_arr[0],
        lower_x_bound,
        upper_x_bound,
        linestyle="--",
        color="red",
        linewidth=linewidth,
    )
    return ax

In [ ]:
sgrna_df_subset["Segregation Length Pixels"] = convert_to_pixels(
    sgrna_df_subset["Segregation Length"], pixel_microns, projection_length_reg
)
sgrna_df_subset["Division Length Pixels"] = convert_to_pixels(
    sgrna_df_subset["Division Length"], pixel_microns, projection_length_reg
)
sgrna_df_subset["Birth Length Pixels"] = convert_to_pixels(
    sgrna_df_subset["Birth Length"], pixel_microns, projection_length_reg
)

In [ ]:
plt.figure(figsize=(8, 3))

varid_list = [4035, 9860, 11246, 12913, 22799, 26540]
nrows = 2
ncols = 3

for i, varid in enumerate(varid_list):
    var_sgrna_df = sgrna_df_subset.loc[varid]
    var_forkplot_df = forkplot_df.loc[varid]
    ax = plt.subplot(nrows, ncols, i + 1)
    ax = make_forkplot_img(
        ax,
        var_forkplot_df,
        var_sgrna_df,
        vmin_perc=2,
        vmax_perc=98,
        img_pad=3,
        linewidth=2,
    )
    ax.set_title(str(varid) + ", " + str(var_sgrna_df["Gene"]))
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

### Figure 4D

In [ ]:
fig = plt.figure(figsize=(3, 3))
ax = plt.subplot(1, 1, 1)
ax = make_forkplot_img(
    ax, var_forkplot_df, var_sgrna_df, vmin_perc=2, vmax_perc=98, img_pad=3, linewidth=2
)
ax.axis("off")
plt.savefig(
    "/home/de64/group/de64/CRISPRi_Libraries/dev_notebooks/2023-04-05_Multi_Dataset_Figures/Nucleoid_Segregation/Example_Forkplot.png",
    transparent=True,
    dpi=500,
    bbox_inches="tight",
)

In [ ]:
dask_controller.shutdown()

### Probability Plots

In [ ]:
def get_prob_plot(
    var_segregation_full_df, tick_label_size=18, label_size=20, n_sem_err=2
):
    length_series = var_segregation_full_df["Length"]
    prob_series = var_segregation_full_df["Probability Segregated"]
    sem_series = var_segregation_full_df["SEM: Probability Segregated"]

    y_err = sem_series * n_sem_err
    # Create the plot
    ax.plot(length_series, prob_series, linewidth=2)
    # Create the envelope
    ax.fill_between(
        length_series, prob_series - y_err, prob_series + y_err, color="gray", alpha=0.5
    )
    # Add labels, title, etc. if needed
    ax.set_xlabel("Length", fontsize=label_size)
    ax.set_ylabel("Prob Seg", fontsize=label_size)
    ax.set_ylim(0, 1)
    plt.tick_params(axis="both", labelsize=tick_label_size, width=3)
    for spine in plt.gca().spines.values():
        spine.set_linewidth(3)
    sns.despine(top=True, right=True)
    # Set x-tick label size
    # Show the plot
    return ax

In [ ]:
plt.figure(figsize=(8, 3))

varid_list = [4035, 9860, 11246, 12913, 22799, 26540]
nrows = 2
ncols = 3

for i, varid in enumerate(varid_list):
    var_segregation_full_df = segregation_full_df.loc[varid]
    ax = plt.subplot(nrows, ncols, i + 1)
    ax.set_xlim(2, 6)
    ax.hlines(0.5, 2, 6, color="black")
    ax = get_prob_plot(
        var_segregation_full_df, tick_label_size=14, label_size=14, n_sem_err=2
    )
    ax.set_title(str(varid) + ", " + str(var_sgrna_df["Gene"]))
plt.tight_layout()
plt.show()

### Nucleoids at Critical Segregation Value

- Spot check looks good!
- Systematically export later

In [ ]:
def lookup_lineage_df(
    lineage_dd,
    file_index_list,
    file_trench_index_list,
    time_range=(0, 1000),
    index_precisions=np.array([3, 2, 3, 0], dtype="uint"),
    index_columns=["File Index", "File Trench Index", "timepoints", "CellID"],
):
    index_precisions = index_precisions.astype("uint")
    rolling_precisions = np.add.accumulate(index_precisions[::-1])
    precision_factors = 10**rolling_precisions

    index_lookup_list = [
        {
            "File Index": file_index_list[i],
            "File Trench Index": file_trench_index_list[i],
        }
        for i in range(len(file_index_list))
    ]
    lineage_df_list = []
    for index_lookup in index_lookup_list:
        init_lower, _ = tr.lookup_parquet_index(
            index_lookup, index_precisions, index_columns
        )
        time_lower_factor = np.array(time_range[0], dtype="uint") * precision_factors[1]
        time_upper_factor = (
            np.array(time_range[1] + 1, dtype="uint") * precision_factors[1]
        )
        lower = init_lower + time_lower_factor
        upper = init_lower + time_upper_factor
        lineage_df = lineage_dd.loc[lower:upper].compute(scheduler="threads")
        lineage_df = lineage_df.set_index(
            ["File Index", "File Trench Index", "timepoints", "CellID"]
        )
        lineage_df_list.append(lineage_df)

    lineage_df = pd.concat(lineage_df_list, axis=0)
    lineage_df = lineage_df.sort_index()
    return lineage_df


def get_variant_img_stack(
    headpath,
    barcode_df,
    sgrna_df_subset,
    lineage_dd,
    varid,
    intensity_channel_name,
    time_range=(0, 1000),
    l_seg_pad=0.1,
    n_examples=100,
    cell_y_pad=20,
):
    variant_file_indices = barcode_df.loc[varid]
    variant_sgrna_df_subset = sgrna_df_subset.loc[varid]
    lineage_df = lookup_lineage_df(
        lineage_dd,
        [item[0] for item in variant_file_indices],
        [item[1] for item in variant_file_indices],
        time_range=time_range,
    )

    l_seg = variant_sgrna_df_subset["Segregation Length"]
    l_seg_upper = l_seg + l_seg_pad
    l_seg_lower = l_seg - l_seg_pad

    lineage_df_crit_length = lineage_df[
        (lineage_df["major_axis_length"] < l_seg_upper)
        & (lineage_df["major_axis_length"] > l_seg_lower)
    ]
    lineage_df_crit_length_subset = lineage_df_crit_length.sample(
        n_examples, replace=False
    )
    cell_img_stack = []
    for file_idx in lineage_df_crit_length_subset.index.get_level_values(0).unique():
        with h5py.File(
            headpath + "/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r"
        ) as infile:
            infile_data = infile[intensity_channel_name][:]
            for file_trench_idx in (
                lineage_df_crit_length_subset.loc[file_idx]
                .index.get_level_values(0)
                .unique()
            ):
                for i, row in (
                    lineage_df_crit_length_subset.loc[file_idx, file_trench_idx]
                    .reset_index()
                    .iterrows()
                ):
                    cell_img = infile_data[
                        file_trench_idx,
                        row["timepoints"],
                        int((row["centroid_y"] / pixel_microns) - cell_y_pad) : int(
                            (row["centroid_y"] / pixel_microns) + cell_y_pad
                        ),
                    ]
                    cell_img_stack.append(cell_img)
    cell_img_stack = np.stack(cell_img_stack)
    return cell_img_stack

In [ ]:
barcode_df = (
    barcode_dd.groupby("oDEPool7_id")
    .apply(
        lambda x: sorted(list(set(tuple(zip(x["File Index"], x["File Trench Index"])))))
    )
    .compute()
    .sort_index()
)
barcode_df = barcode_df.apply(lambda x: eval(x))

#### Picking Variants

In [ ]:
l_seg_pad = 0.1
varid = 12913
n_examples = 100
cell_y_pad = 20
time_range = (49, 60)
intensity_channel_name = "mCherry"

cell_img_stack = get_variant_img_stack(
    headpath,
    barcode_df,
    sgrna_df_subset,
    lineage_dd,
    varid,
    intensity_channel_name,
    time_range=time_range,
    l_seg_pad=l_seg_pad,
    n_examples=n_examples,
    cell_y_pad=cell_y_pad,
)

In [ ]:
ncols = 10
nrows = 10

n_imgs = ncols * nrows

fig = plt.figure(figsize=(10, 10))

for img_idx in range(n_imgs):
    plt.subplot(ncols, nrows, img_idx + 1)
    plt.imshow(cell_img_stack[img_idx])

plt.show()

In [ ]:
dask_controller.shutdown()